# Notebook 08 — Wozformer-RAG: Retrieval, not generation

*Architectural pivot. Semantic output by construction.*

## What we're doing and why

The transformer in nb07b produces **real words plus made-up tokens** because it's still trying to *generate* novel text from a tiny capacity. That's the hardest problem in NLP, attempted on the smallest possible silicon.

This notebook does the opposite. Instead of generating, we **retrieve**:

1. Store ~300 real Shakespeare lines in EEPROM.
2. Train a tiny **sentence encoder** that maps any text → 8-dim vector.
3. Embed every stored line; ship the embeddings alongside the lines.
4. At runtime: embed the user's prompt, find the nearest stored line by cosine similarity, emit it character-by-character.

Output is **always real Shakespeare**, because the only thing the 6502 can output is what's in its phrase library. The 'model' is just deciding *which* line to pull.

### Why retrieval works at this scale

Generation requires modeling `P(next_token | context)` *accurately* for every possible context. That's a fundamentally information-hungry problem — small models lose.

Retrieval only requires the embedding to *rank* candidates correctly. The encoder doesn't need to know what every token *means*, just enough geometry to place semantically-similar phrases near each other in 8-dim space. **Ranking is much easier than predicting.**

At our scale (~2 KB encoder, ~300 candidates), even a mean-pooled token embedding with one contrastive-trained projection produces useful retrieval. Sentence-BERT is the same idea at large scale.

### The contrastive learning trick

How do you train an encoder *without* labels?

**Positive pairs**: lines that appear adjacent in the corpus (line N, line N+1). They're semantically related by virtue of being part of the same dialogue.

**Negative pairs**: random other lines from the corpus.

**Loss (InfoNCE)**: for a batch of B positive pairs, treat it as a B-way classification problem — given pair member A, the correct match is its true partner B (out of all B candidates in the batch). Cross-entropy loss on the similarity scores.

This forces the encoder to learn: *what makes two lines belong together?* The answer turns out to be "shared topic, shared character, shared theme" — exactly what we want for retrieval.

### What's new (vs nb07b)

- **Sentence encoder architecture** (mean-pool + projection, not a generative model)
- **Contrastive training** (positive/negative pairs, not next-token prediction)
- **A new binary format** (`wozformer_rag.bin`) containing the database + encoder + tokenizer


## Cell 1 — Setup

We re-use the BPE tokenizer from nb07b. To keep this notebook standalone we re-train the BPE here (it's fast) instead of loading from disk.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct, random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
random.seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text().lower()
print(f'corpus: {len(text):,} chars')


## Cell 2 — BPE tokenizer (re-trained, same as nb07b)

Identical algorithm to nb07b. Pasted here for self-containment.


In [ ]:
def train_bpe(text, num_merges):
    EOW = '</w>'
    word_freq = Counter(tuple(list(w) + [EOW]) for w in text.split())
    word_lists = {w: list(w) for w in word_freq}
    merges = []
    for step in range(num_merges):
        pair_counts = Counter()
        for w, freq in word_freq.items():
            symbols = word_lists[w]
            for i in range(len(symbols)-1):
                pair_counts[(symbols[i], symbols[i+1])] += freq
        if not pair_counts: break
        best, _ = pair_counts.most_common(1)[0]
        new_tok = best[0] + best[1]
        merges.append((best, new_tok))
        for w in word_freq:
            symbols = word_lists[w]
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == best:
                    new_symbols.append(new_tok); i += 2
                else:
                    new_symbols.append(symbols[i]); i += 1
            word_lists[w] = new_symbols
    vocab_set = set()
    for w in word_freq:
        vocab_set.update(word_lists[w])
        vocab_set.update(w)
    return merges, sorted(vocab_set)

EOW = '</w>'
VOCAB_SIZE = 128
print('training BPE (88 merges)...')
merges, vocab = train_bpe(text, 88)

all_toks = ['<unk>', '<pad>'] + sorted(vocab)
while len(all_toks) < VOCAB_SIZE: all_toks.append(f'<pad{len(all_toks)}>')
all_toks = all_toks[:VOCAB_SIZE]
itos = all_toks
stoi = {t: i for i, t in enumerate(itos)}
PAD_ID = stoi['<pad>']
UNK_ID = stoi['<unk>']
print(f'vocab size = {len(itos)}  (PAD={PAD_ID}, UNK={UNK_ID})')

def encode_word(word):
    symbols = list(word) + [EOW]
    for (a, b), merged in merges:
        i = 0; new_symbols = []
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i]==a and symbols[i+1]==b:
                new_symbols.append(merged); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return [stoi.get(s, UNK_ID) for s in symbols]

def encode(text):
    ids = []
    for w in text.split():
        ids.extend(encode_word(w))
    return ids

def decode(ids):
    s = ''.join(itos[i] for i in ids if i != PAD_ID)
    return s.replace(EOW, ' ').strip()


## Cell 3 — Curate the phrase database (300 lines)

### Selection criteria

We need 300 lines that are:

1. **Long enough to be meaningful** — drop short interjections like `'Aye.'`, `'Ho!'`.
2. **Short enough to fit** — drop super-long speeches that would consume the database.
3. **Diverse** — avoid 50 near-duplicate `'Yes, my lord.'` variants.

### Practical filter

- Split corpus on newlines.
- Keep lines between **40 and 160 chars** (one well-formed sentence).
- De-duplicate (case-insensitive exact match).
- Sort by length, take a stratified sample so we get short, medium, and long lines proportionally.


In [ ]:
# Use the ORIGINAL-case corpus for the database (more visually interesting on the LCD)
orig_text = Path('../data/tinyshakespeare.txt').read_text()
raw_lines = orig_text.split('\n')

# Filter
candidates = []
seen = set()
for ln in raw_lines:
    ln = ln.strip()
    if not (40 <= len(ln) <= 160): continue
    if ln.lower() in seen: continue
    seen.add(ln.lower())
    candidates.append(ln)

print(f'candidate lines after filter: {len(candidates):,}')

# Stratified sample: 100 short (40-70), 100 medium (70-110), 100 long (110-160)
def bucket(ln):
    if len(ln) <= 70: return 'short'
    if len(ln) <= 110: return 'medium'
    return 'long'

by_bucket = {'short': [], 'medium': [], 'long': []}
for ln in candidates:
    by_bucket[bucket(ln)].append(ln)

for b, lns in by_bucket.items():
    print(f'  {b:>6}: {len(lns):,} candidates')

# Take 100 random from each bucket
DB_SIZE = 300
per_bucket = DB_SIZE // 3
database = []
for b in ('short', 'medium', 'long'):
    random.shuffle(by_bucket[b])
    database.extend(by_bucket[b][:per_bucket])

random.shuffle(database)
print(f'\ndatabase: {len(database)} lines')
print(f'total chars: {sum(len(l) for l in database):,}')
print('\nfirst 5:')
for ln in database[:5]:
    print(f'  ({len(ln):>3} ch) {ln!r}')


## Cell 4 — Encode the database with BPE

Each phrase becomes a list of token ids. We **don't** pad here — variable lengths are fine because we'll mean-pool over actual tokens.

For the binary file we'll store each phrase as `(length_byte, token_bytes...)`. The 6502 reads `length`, then `length` bytes of token IDs, decodes via the vocab table.


In [ ]:
db_encoded = [encode(line.lower()) for line in database]

lengths = [len(t) for t in db_encoded]
print(f'tokens per line: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}')
print(f'total tokens in database: {sum(lengths):,}')
print(f'estimated database bytes (1 byte per token + length headers): {sum(lengths) + len(db_encoded):,}')

# Ensure no line is > 255 tokens (so length fits in a byte)
assert max(lengths) < 256, 'phrase too long for 1-byte length prefix'

print('\nexample:')
print(f'  original: {database[0]!r}')
print(f'  encoded:  {db_encoded[0]}')
print(f'  decoded:  {decode(db_encoded[0])!r}')


## Cell 5 — The sentence encoder

### Architecture

```
token_ids (B, T)
  ▼ token_embed (V × d_model)
(B, T, d_model)
  ▼ mask + mean-pool over T (ignoring <pad> tokens)
(B, d_model)
  ▼ Linear(d_model, d_embed)
(B, d_embed)
  ▼ L2-normalize
(B, d_embed)  ← unit vectors on the d_embed-1 sphere
```

### Why so simple?

At our scale (300 phrases, 8-dim embedding), complexity hurts. The 6502 has to compute this at runtime, so every parameter costs cycles. The simplest architecture that produces a meaningful embedding is the right one.

Key choices:

- **`d_model = 16`** — bigger than nb07b's 32 because we're not constrained by attention (no quadratic costs). Bigger token embeddings → better discrimination.
- **`d_embed = 8`** — final sentence vector dim. Small for fast 6502 inner products. Large enough to separate 300 lines in space.
- **L2-normalize** — turns dot product = cosine similarity. The 6502 doesn't need to compute square roots at retrieval time.

### What gets shipped to the 6502

- `token_embed.weight` (V × d_model = 128 × 16 = 2,048 floats → 2 KB int8)
- `project.weight` (d_model × d_embed = 16 × 8 = 128 floats → 128 B int8)
- `project.bias` (8 floats → 8 B int8)

Total encoder: **~2.2 KB**. Cycles per query: ~T×d_model (token lookups) + d_model×d_embed (projection) = at T=10, ~160 + 128 = ~288 mults. Negligible.


In [ ]:
D_MODEL = 16
D_EMBED = 8

class SentenceEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, d_embed, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.project = nn.Linear(d_model, d_embed)

    def forward(self, idx):
        # idx: (B, T) longs, possibly padded with pad_id
        mask = (idx != self.pad_id).float().unsqueeze(-1)  # (B, T, 1) — 1 where real token, 0 where pad
        x = self.token_embed(idx) * mask                    # (B, T, d_model), pads zeroed
        # Mean-pool over real tokens (avoid div-by-zero with clamp)
        token_counts = mask.sum(dim=1).clamp(min=1.0)       # (B, 1)
        pooled = x.sum(dim=1) / token_counts                # (B, d_model)
        e = self.project(pooled)                            # (B, d_embed)
        return F.normalize(e, dim=-1)                       # L2-normalize

encoder = SentenceEncoder(VOCAB_SIZE, D_MODEL, D_EMBED, PAD_ID).to(device)
print(f'encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}')
for n, p in encoder.named_parameters():
    print(f'  {n:30s} {tuple(p.shape)}')


## Cell 6 — Contrastive training data

### Positive pairs from corpus adjacency

We treat **consecutive lines in the corpus** as positive pairs. The intuition: in a play, line N and line N+1 are part of the same dialogue, likely share characters, topic, mood. They *should* be near each other in embedding space.

This is **self-supervised** — no human labels. Same trick that powers SimCSE and works embarrassingly well in NLP.

### Filter for quality pairs

We use the same filter as the database: both lines 40-160 chars. Otherwise positive pairs include junk like `'Aye.'`/`'No, my lord.'` which is too noisy.

### Negative pairs are free

With InfoNCE loss, **other items in the batch ARE the negatives**. So we just need positive pairs and a reasonable batch size. Batch=64 → 63 negatives per example.


In [ ]:
# Build positive pairs (consecutive lines that both pass the filter)
raw_lines2 = [ln.strip() for ln in orig_text.split('\n')]
pairs = []
for i in range(len(raw_lines2) - 1):
    a, b = raw_lines2[i], raw_lines2[i+1]
    if 20 <= len(a) <= 160 and 20 <= len(b) <= 160:
        pairs.append((a, b))

print(f'positive pairs: {len(pairs):,}')
print('\nexample pair:')
a, b = pairs[0]
print(f'  A: {a!r}')
print(f'  B: {b!r}')

# Pre-encode all pairs (lowercased to match training data)
def encode_pad(text, max_len=24):
    ids = encode(text.lower())[:max_len]
    while len(ids) < max_len:
        ids.append(PAD_ID)
    return ids

MAX_LEN = 24
pairs_enc = [(encode_pad(a, MAX_LEN), encode_pad(b, MAX_LEN)) for a, b in pairs]
pairs_tensor_A = torch.tensor([p[0] for p in pairs_enc], dtype=torch.long)
pairs_tensor_B = torch.tensor([p[1] for p in pairs_enc], dtype=torch.long)
print(f'\ntensor shape: {pairs_tensor_A.shape}  (N, T)')


## Cell 7 — InfoNCE loss and training loop

### InfoNCE in 3 lines

For a batch of B (A_i, B_i) positive pairs:

1. Encode A and B → unit vectors `e_A`, `e_B` of shape (B, d_embed).
2. Compute similarity matrix `S = e_A @ e_B.T / tau` of shape (B, B). Entry `S[i, j]` is the similarity between A_i and B_j.
3. Cross-entropy: treat row `i` as a classification problem where the correct class is `i` (because A_i's true partner is B_i). Loss = `F.cross_entropy(S, range(B))`.

This forces the encoder to make `S[i, i] > S[i, j]` for all `j ≠ i` — i.e. the true pair beats all distractors. **Symmetric version** also requires the same for B-side queries; we'll do both and average.

### Temperature `tau`

`tau = 0.05` is standard. Smaller tau → sharper softmax → harder to learn but better separation. We use 0.07 which is what CLIP and SimCSE use.


In [ ]:
BATCH_SIZE = 64
LR = 3e-3
N_STEPS = 4000
EVAL_EVERY = 200
TAU = 0.07

def sample_batch(n=BATCH_SIZE):
    ix = torch.randint(0, len(pairs_tensor_A), (n,))
    return pairs_tensor_A[ix].to(device), pairs_tensor_B[ix].to(device)

def info_nce_loss(eA, eB, tau=TAU):
    # eA, eB: (B, d_embed), L2-normalized
    sim = eA @ eB.T / tau   # (B, B)
    targets = torch.arange(sim.size(0), device=sim.device)
    loss_a = F.cross_entropy(sim, targets)
    loss_b = F.cross_entropy(sim.T, targets)
    return (loss_a + loss_b) / 2

opt = torch.optim.AdamW(encoder.parameters(), lr=LR, weight_decay=0.01)
history = []
best_val = float('inf')
best_state = None

# Use last 5% of pairs as val
VAL_N = max(1, len(pairs_enc) // 20)
val_A = pairs_tensor_A[-VAL_N:].to(device)
val_B = pairs_tensor_B[-VAL_N:].to(device)

@torch.no_grad()
def eval_val(n_chunks=4):
    encoder.eval()
    losses = []
    chunk_size = len(val_A) // n_chunks
    for i in range(n_chunks):
        a = val_A[i*chunk_size:(i+1)*chunk_size]
        b = val_B[i*chunk_size:(i+1)*chunk_size]
        eA, eB = encoder(a), encoder(b)
        losses.append(info_nce_loss(eA, eB).item())
    encoder.train()
    return sum(losses) / len(losses)

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        vl = eval_val()
        history.append((step, vl))
        marker = ''
        if vl < best_val:
            best_val = vl
            best_state = {k: v.detach().clone() for k, v in encoder.state_dict().items()}
            marker = '  <-- new best'
        print(f'step {step:>4} | val loss {vl:.4f}{marker}')
    a, b = sample_batch()
    eA, eB = encoder(a), encoder(b)
    loss = info_nce_loss(eA, eB)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

encoder.load_state_dict(best_state)
print(f'\nloaded best (val {best_val:.4f}).')


## Cell 8 — Encode the database and demo retrieval in Python

Run all 300 lines through the trained encoder, store the 300×8 embedding matrix. Then test: given an arbitrary prompt, return the top-3 nearest database lines.


In [ ]:
@torch.no_grad()
def embed_text(text):
    ids = encode_pad(text.lower(), MAX_LEN)
    t = torch.tensor([ids], dtype=torch.long, device=device)
    return encoder(t).squeeze(0)   # (d_embed,)

# Encode all database lines
db_embeds = torch.zeros(len(database), D_EMBED, device=device)
for i, line in enumerate(database):
    db_embeds[i] = embed_text(line)

def retrieve(query, k=3):
    q = embed_text(query)                       # (d_embed,)
    sims = db_embeds @ q                        # (N,)
    top = torch.topk(sims, k=k)
    return [(database[i.item()], sims[i].item()) for i in top.indices]

prompts = [
    'O Romeo, where art thou?',
    'The king is dead',
    'I love you, my lord',
    'War approaches',
    'A traitor in our midst',
]
for p in prompts:
    print(f'\nPROMPT: {p!r}')
    for line, score in retrieve(p, k=3):
        print(f'  [{score:+.3f}] {line!r}')


## Cell 9 — Quantize encoder + embeddings to int8

Same PTQ approach as nb07. The encoder's `token_embed` and `project` are quantized to int8 with per-tensor scales. The 300 phrase embeddings are also quantized — separately, with their own scale.

### Why per-tensor for the phrase embeddings?

All 300 are normalized to unit norm, so they all live on the d=8 sphere. Per-tensor scale captures the range tightly (everything is ≤ 1 in magnitude). One scale suffices.


In [ ]:
def quantize_symmetric(t):
    max_abs = t.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(t, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    qt = torch.round(t / scale).clamp(-128, 127).to(torch.int8)
    return qt, scale

def dequantize(qt, s): return qt.to(torch.float32) * s

q_table = {}
for name, p in encoder.named_parameters():
    if p.dim() < 1: continue
    qp, s = quantize_symmetric(p.data)
    q_table[name] = (qp, s)
    p.data.copy_(dequantize(qp, s))

# Re-encode the database with the quantized encoder
with torch.no_grad():
    for i, line in enumerate(database):
        db_embeds[i] = embed_text(line)

# Quantize the phrase embeddings
db_qe, db_scale = quantize_symmetric(db_embeds)
print(f'phrase embeddings: shape={tuple(db_qe.shape)}, scale={db_scale:.6f}')

# Retrieval check after quantization
print('\nRetrieval after int8 quantization:')
for p in prompts[:3]:
    print(f'  PROMPT: {p!r}')
    for line, score in retrieve(p, k=1):
        print(f'    -> [{score:+.3f}] {line!r}')


## Cell 10 — Pack `wozformer_rag.bin`

### File format

```
offset  size  contents
------  ----  --------
0       4     magic 'WRAG'
4       1     version (1)
5       1     vocab_size (128)
6       1     d_model    (16)
7       1     d_embed    (8)
8       2     num_merges (uint16 LE)
10      2     num_phrases (uint16 LE) = 300
12      ...   vocab strings (length-prefixed utf-8, 128 entries)
...     ...   merges (each: len_a, a, len_b, b, len_merged, merged)

--- encoder weights (int8 with scales) ---
...     4     scale (f32)
...     V×d_model   token_embed int8
...     4     scale (f32)
...     d_model×d_embed   project.weight int8
...     4     scale (f32)
...     d_embed   project.bias int8

--- phrase index (300 entries) ---
...     4     scale (f32) for ALL phrase embeddings
...     300×d_embed   phrase embeddings (int8, row-major)
...     300×2  phrase offsets (uint16 LE) into phrase storage
...     300×1  phrase lengths in tokens (uint8)

--- phrase storage ---
...     Σlengths   token IDs (uint8) for each phrase, concatenated
```

### Why this layout?

The 6502 firmware does, per query:

1. Read prompt token IDs from Arduino.
2. Look up each token's embedding row (token_embed offset).
3. Sum + average + project + normalize → 8-byte prompt embedding.
4. Loop over 300 phrase embeddings, compute dot product (8 mults each).
5. Pick the index with the largest dot product.
6. Read that phrase's offset + length from the index.
7. Stream token IDs from phrase storage, decode via vocab table, send to Arduino → LCD.

Each step is **just memory reads + small math**. No softmax. No layernorm. The whole thing is way simpler than transformer inference.


In [ ]:
export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer_rag.bin'

buf = bytearray()
buf += b'WRAG'
buf += bytes([1, VOCAB_SIZE, D_MODEL, D_EMBED])
buf += struct.pack('<HH', len(merges), len(database))

# Vocab strings
for tok in itos:
    b = tok.encode('utf-8')
    assert len(b) < 256
    buf += bytes([len(b)]) + b

# Merges
for (a, b), merged in merges:
    for piece in (a, b, merged):
        pb = piece.encode('utf-8')
        buf += bytes([len(pb)]) + pb

# Encoder weights
for name in ['token_embed.weight', 'project.weight', 'project.bias']:
    qp, s = q_table[name]
    buf += struct.pack('<f', s)
    buf += qp.cpu().numpy().tobytes()

# Phrase embeddings (one scale for all 300)
buf += struct.pack('<f', db_scale)
buf += db_qe.cpu().numpy().tobytes()

# Phrase offsets and lengths
all_tokens = bytearray()
offsets = []
lengths = []
for enc in db_encoded:
    offsets.append(len(all_tokens))
    lengths.append(len(enc))
    all_tokens += bytes(enc)   # each token id fits in uint8 (vocab=128)

# Sanity
assert max(offsets) < 65536, 'offset overflow'
assert max(lengths) < 256

# Offsets (uint16 LE)
for off in offsets:
    buf += struct.pack('<H', off)
# Lengths (uint8)
buf += bytes(lengths)

# Phrase storage (concatenated token IDs)
buf += bytes(all_tokens)

out_path.write_bytes(buf)

BUDGET = 32 * 1024
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM budget (32 KB): {100*len(buf)/BUDGET:.1f}% used')
print(f'headroom: {BUDGET - len(buf):,} bytes')

# Breakdown
print('\nbyte breakdown (estimated):')
print(f'  vocab strings:     ~{sum(1+len(t.encode()) for t in itos):,}')
print(f'  merges:            ~{sum(3 + len(a.encode()) + len(b.encode()) + len(m.encode()) for (a,b),m in merges):,}')
enc_bytes = sum(q_table[n][0].numel() + 4 for n in ['token_embed.weight','project.weight','project.bias'])
print(f'  encoder weights:   ~{enc_bytes:,}')
print(f'  phrase embeddings: ~{db_qe.numel() + 4:,}')
print(f'  phrase index:      ~{2*len(offsets) + len(lengths):,}')
print(f'  phrase storage:    ~{len(all_tokens):,}')


## Cell 11 — Cycle budget

Per-query cost on the 6502:

- **Embed prompt**: ~T (prompt tokens) lookups + d_model × d_embed (projection) ≈ ~150 mults.
- **Score 300 phrases**: 300 × d_embed (dot products) = 300 × 8 = 2,400 mults.
- **Total: ~2,550 mults × 80 cycles ≈ 200,000 cycles = 200 ms per query.**

Then streaming the chosen phrase is **zero compute** — just EEPROM reads + LCD writes.

Compare to nb07b's ~2,000 ms *per token* of a generated phrase. RAG is 10× faster *and* produces real Shakespeare.


In [ ]:
T_prompt = MAX_LEN
embed_cost = T_prompt * D_MODEL + D_MODEL * D_EMBED + D_EMBED  # lookups + project + normalize approx
score_cost = len(database) * D_EMBED
total = embed_cost + score_cost
print(f'prompt embed:     {embed_cost:>5,} mults')
print(f'score 300 phrases: {score_cost:>5,} mults')
print(f'total per query:  {total:>5,} mults')
print(f'@80 cycles/mul:  {total*80:>7,} cycles = {total*80/1e6*1000:.1f} ms @ 1 MHz')
print('then streaming chosen phrase ~50 chars × ~1ms serial = ~50ms')
print(f'\nUser-perceived latency: ~{(total*80/1e6+0.05)*1000:.0f} ms from prompt to first output char.')


## Cell 12 — Save checkpoint for the C reference


In [ ]:
ckpt = Path('../export/wozformer_rag.pt')
torch.save({
    'vocab_size': VOCAB_SIZE, 'd_model': D_MODEL, 'd_embed': D_EMBED, 'pad_id': PAD_ID,
    'encoder_state': encoder.state_dict(),
    'database': database,
    'db_encoded': db_encoded,
    'db_embeddings_int8': db_qe.cpu().numpy().tolist(),
    'db_embedding_scale': db_scale,
    'itos': itos,
    'merges': [(list(p), m) for p, m in merges],
    'best_val_infonce': best_val,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — semantic output by construction

### What you produced

- `export/wozformer_rag.bin` — ~25 KB, contains 300 real Shakespeare lines + a 2 KB neural retriever + BPE tokenizer.
- `export/wozformer_rag.pt` — Python checkpoint for the C reference.

### Architecture vs nb07b

| | nb07b (transformer) | **nb08 (RAG)** |
|---|---|---|
| Inference type | Generation (predict next token) | Retrieval (rank stored phrases) |
| Per-token cost | ~17,000 mults (~1,400 ms) | N/A — phrase emitted whole |
| Per-query cost | (no concept of query) | ~2,500 mults (~200 ms) |
| Output type | Generated text (real words + made-up ones) | **Real Shakespeare lines only** |
| Semantic? | Locally yes, globally no | **Yes by construction** |
| 6502 firmware complexity | Hard (softmax, layernorm, KV cache) | **Easy (lookup + dot products)** |

### The honest tradeoff

- **RAG wins on output quality and latency.** Every reply is real Shakespeare. Inference is 10× faster.
- **Transformer wins on flexibility.** It can generate text for prompts no stored phrase matches.
- **RAG is fundamentally retrieval; transformer is fundamentally generation.** Different problems, different tools.

### What this means for the project

You now have **three deployment options**:

1. **Ship `wozformer_v2.bin`** (transformer): the most architecturally interesting / hardest to implement on 6502 — *the original challenge*.
2. **Ship `wozformer_rag.bin`** (RAG): the highest-quality output / easiest 6502 implementation — *the pragmatist's pick*.
3. **Ship both, let the user toggle** (one mode switch on the breadboard): you get to demo both architectures running on the same chip. Best for showing off / teaching.

### Next: C reference

For whichever path you choose, the next milestone is the C reference — a laptop-runnable oracle that exactly matches what the 6502 will compute. Without it, debugging 6502 assembly is impossible.

Onwards.
